# Parking Sign Detection - A/B Test: Version A (Negative Annotations ON, Augmentation OFF)

Isolating the effect of negative samples by disabling aggressive augmentation.

**Variable under test:** Negative annotations (background crops with empty labels)
**Control:** Augmentation OFF

**Dataset:** ~8,600+ images, 1 class (parking_sign)
- Combined from: parking-sign-coco (Roboflow) + sf-parking-signs (Figure Eight)
- All images resized to 640x640
- **10 sign-sized negative background crops per training image (INCLUDED)**
- Train/Val/Test split: 80/10/10

**Model:** YOLO11m (Medium)

**Training:** 30 epochs, cosine LR decay, 2 GPUs, **minimal augmentation**

## 1. Setup

In [ ]:
# Fix numpy version first (torchvision requires numpy<2)
!pip uninstall numpy -y -q
!pip install "numpy<2" -q

# Then install ultralytics
!pip install ultralytics -q

# Restart runtime after this cell if needed

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import yaml
import os

# Kaggle dataset path (nested directory structure confirmed by user)
DATASET_PATH = Path("/kaggle/input/parking-sign-detection-coco-dataset/parking-sign-detection-coco-dataset")
OUTPUT_PATH = Path("/kaggle/working")
DATA_YAML = DATASET_PATH / "data.yaml"

print(f"Dataset Path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

if DATASET_PATH.exists():
    print(f"Contents: {list(DATASET_PATH.iterdir())}")
    
    # Parse data.yaml
    with open(DATA_YAML) as f:
        data_config = yaml.safe_load(f)
    
    print("Original config:", data_config)
    
    # Update paths to use absolute paths
    new_config = data_config.copy()
    
    for split in ["train", "val", "test"]:
        keys = [split]
        if split == 'val': keys.append('valid')
        
        for k in keys:
            if k in new_config:
                folder_name = "valid" if split == "val" else split
                
                img_path = DATASET_PATH / folder_name / "images"
                if not img_path.exists():
                    img_path = DATASET_PATH / folder_name
                
                if img_path.exists():
                    new_config[k] = str(img_path)
                    print(f"Resolved {k} to {img_path}")
                else:
                    print(f"WARNING: Could not find image path for {k} at {img_path}")

    new_config['path'] = str(DATASET_PATH)
    
    FIXED_DATA_YAML = OUTPUT_PATH / "data.yaml"
    with open(FIXED_DATA_YAML, "w") as f:
        yaml.dump(new_config, f)
    
    DATA_YAML = FIXED_DATA_YAML
    print(f"\nFixed data.yaml written to {DATA_YAML}")
    !cat {DATA_YAML}
else:
    print("ERROR: DATASET_PATH not found. Please check directory structure in /kaggle/input")
    !ls -R /kaggle/input | head -n 20

## 2. Dataset Verification

In [ ]:
# Count images in each split
for split in ["train", "valid", "test"]:
    try:
        folder_name = split
        if split == 'val' and not (DATASET_PATH / split).exists():
             folder_name = 'valid'
             
        img_dir = DATASET_PATH / folder_name / "images"
        label_dir = DATASET_PATH / folder_name / "labels"
        
        if img_dir.exists():
            n_images = len(list(img_dir.glob("*.jpg")))
            n_labels = len(list(label_dir.glob("*.txt")))
            print(f"{split} ({folder_name}): {n_images} images, {n_labels} labels")
        else:
            print(f"{split}: Directory not found at {img_dir}")
    except Exception as e:
        print(f"Error checking {split}: {e}")

## 3. Training Configuration

### Version A: Negative Annotations ON, Augmentation OFF
Isolates the effect of negative samples by disabling all aggressive augmentation.
Dataset includes background crops with empty label files (negative annotations).
Only basic flips, slight translate, and mild HSV jitter are kept.

In [ ]:
RUN_NAME = "parking_sign_negann_noaug"

TRAIN_PARAMS = {
    "data": str(DATA_YAML),
    "epochs": 30,
    "imgsz": 640,
    "batch": 16,
    "workers": 4,
    "patience": 15,
    "save_period": 10,
    "project": str(OUTPUT_PATH / "runs"),
    "name": RUN_NAME,
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
    # Multi-GPU (DDP)
    "device": "0,1",
    # Cosine LR decay
    "cos_lr": True,
    "lr0": 0.005,
    "lrf": 0.005,
    # MINIMAL augmentation - OFF
    "mosaic": 0.0,          # OFF
    "mixup": 0.0,           # OFF
    "copy_paste": 0.0,      # OFF
    "degrees": 0.0,         # OFF
    "translate": 0.1,       # Keep minimal
    "scale": 0.0,           # OFF
    "shear": 0.0,           # OFF
    "perspective": 0.0,     # OFF
    "fliplr": 0.5,          # Keep - harmless
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.3,
    "hsv_v": 0.2,
}

print(f"Training config: {len(TRAIN_PARAMS)} parameters")
print(f"Model: yolo11m | Epochs: {TRAIN_PARAMS['epochs']}, Patience: {TRAIN_PARAMS['patience']}")
print(f"Image size: {TRAIN_PARAMS['imgsz']} | Device: {TRAIN_PARAMS['device']} (2 GPUs)")
print(f"Negative annotations: ON | Augmentation: OFF")

## 4. Train Model

In [ ]:
print("="*60)
print("Training Version A: Negative Annotations ON, Augmentation OFF")
print("="*60)

model = YOLO("yolo11m.pt")
train_results = model.train(**TRAIN_PARAMS)

## 5. Training Results

In [ ]:
# Show training curves
from IPython.display import Image, display

results_img = OUTPUT_PATH / "runs" / RUN_NAME / "results.png"
if results_img.exists():
    print("Training curves:")
    display(Image(filename=str(results_img)))
else:
    print(f"Results image not found at {results_img}")

## 6. Evaluate Best Model

In [ ]:
# Load best model
best_model_path = OUTPUT_PATH / "runs" / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(best_model_path)

print(f"Loaded model from: {best_model_path}")

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(DATA_YAML), split="test")

print(f"\nTest Set Results:")
print(f"  mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP50-95: {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

In [ ]:
# Show confusion matrix
confusion_img = OUTPUT_PATH / "runs" / RUN_NAME / "confusion_matrix.png"
if confusion_img.exists():
    print("Confusion matrix:")
    display(Image(filename=str(confusion_img)))

## 7. Export Best Model

In [ ]:
# Export to ONNX for deployment
print("Exporting best model to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print(f"Exported to: {onnx_path}")

# Also save PyTorch format to output root
final_model_path = OUTPUT_PATH / f"parking_sign_detector_{RUN_NAME}.pt"
shutil.copy(best_model_path, final_model_path)
print(f"PyTorch model: {final_model_path}")

## 8. Results Summary

### Version A: Negative Annotations ON, Augmentation OFF
- **Epochs:** 30, patience 15
- **Negative annotations:** ON (background crops with empty labels included in dataset)
- **Augmentation:** OFF (only fliplr + slight translate + mild HSV)
- **Hypothesis:** Negative samples alone provide FP reduction; loss curve should be smooth and decreasing

### Compare with Version B
Run `parking_sign_training_fullaug.ipynb` which has the opposite config:
- Negative annotations OFF (filtered out) + Augmentation ON (aggressive)

Compare:
1. Loss curves (smooth vs oscillatory)
2. mAP50, mAP50-95 on test set
3. Decide which variable caused instability

In [ ]:
# Print final summary
print("\n" + "="*60)
print("VERSION A (NEG ANN ON, AUG OFF) TRAINING COMPLETE")
print("="*60)
print(f"\nRun name: {RUN_NAME}")
print(f"Output files:")
print(f"  - {final_model_path}")
print(f"  - {OUTPUT_PATH / 'runs' / RUN_NAME / 'results.csv'}")